[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Math/Optimization/Optimization.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Optimization

Every 'fit', 'train', and 'design' in this curriculum is secretly `argmin`. Four sessions on why gradient methods work, when they're guaranteed to, what constraints do, and why stochastic noise is a feature — the theory under [ANN's](../../Intro_Mach_Learn/Intro_ANN/Intro_ANN.ipynb) backprop loop and [Adaptive Filtering's](../../Intro_Time_Series/Intro_AdFilt_APA.ipynb) LMS.

## 0. Introduction

The object of study: $\min_x f(x)$. Three questions organize the course — *when is the minimum unique?* (convexity), *how fast do we get there?* (rates), *what if $x$ is constrained?* (Lagrange).

## 1. Pre-requisites

[Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) — especially S3 (eigenvalues) and S5 (matrix calculus). Multivariable calculus at the chain-rule level.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *Convexity* (~35 min)
**Goal:** recognize the functions where local = global; test convexity via the Hessian.
**Builds on:** [Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb). &nbsp; **Feeds into:** Session 2 (gradient descent).

---

## 2. Convex Functions

💡 **Intuition.** A convex function is a **bowl**: the chord between any two points sits above the graph. The consequence worth the whole session: *any local minimum is the global minimum* — a downhill walker cannot get trapped. Convex problems are the ones optimization can truly promise to solve; everything else (deep learning included) lives on borrowed intuition from this case.

**Definition.** $f$ is convex if for all $x, y$ and $\theta \in [0,1]$:
$$f(\theta x + (1-\theta) y) \le \theta f(x) + (1-\theta) f(y)$$

**Second-order test.** Twice-differentiable $f$ is convex iff its Hessian $\nabla^2 f \succeq 0$ (all eigenvalues $\ge 0$) everywhere — the bowl curves up along every axis. For a quadratic $f = \tfrac12 x^T S x - b^T x$, the Hessian *is* $S$: convex iff $S \succeq 0$, strictly (unique minimum) iff $S \succ 0$.

**Proof that local ⇒ global (convex case).** Let $x^\*$ be a local min and suppose $f(y) < f(x^\*)$ for some $y$. Points $\theta y + (1-\theta)x^\*$ approach $x^\*$ as $\theta \to 0$, and convexity gives $f(\theta y + (1-\theta)x^\*) \le \theta f(y) + (1-\theta) f(x^\*) < f(x^\*)$ — arbitrarily close points with lower value, contradicting local minimality. $\blacksquare$

In [ ]:
# Convex vs nonconvex, and what a walker experiences

# YOUR CODE HERE


**Convexity zoo you already use:** least squares ($A^TA \succeq 0$), cross-entropy of a linear model, every norm ($\|\cdot\|_1, \|\cdot\|_2$), max of convex functions. **Not convex:** neural network losses — yet Session 4 explains why training works anyway.

---
### 🕐 Session 2 of 4 — *Gradient Descent & Rates* (~40 min)
**Goal:** prove how fast GD converges on smooth/strongly-convex problems; see conditioning bite.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (constraints).

---

## 3. Gradient Descent

💡 **Intuition.** GD's guarantee needs two numbers: $L$ (curvature never exceeds $L$ — you can trust the slope for a step of about $1/L$) and $\mu$ (curvature never below $\mu$ — the bowl never flattens out). Their ratio $\kappa = L/\mu$, the **condition number**, is the villain of the course: it is the elongation of the bowl, and error shrinks by a factor $\approx (1 - 1/\kappa)$ per step. Round bowl ⇒ sprint; canyon ⇒ zigzag crawl. For quadratics, $L$ and $\mu$ are just the extreme eigenvalues from [Linear Algebra S3](../Linear_Algebra/Linear_Algebra.ipynb).

**Theorem (rate, stated).** For $L$-smooth, $\mu$-strongly-convex $f$, GD with step $\eta = 1/L$ satisfies
$$\|x_k - x^*\|^2 \le \big(1 - \tfrac{\mu}{L}\big)^k \, \|x_0 - x^*\|^2$$
— *linear convergence*: a fixed fraction of the remaining error removed per step. (Proof for quadratics is a two-line eigen-argument: the error multiplies by $I - \eta S$, whose eigenvalues are $1 - \eta \lambda_i$.)

In [ ]:
# Watch κ control the speed, exactly as the theorem says

# YOUR CODE HERE


**What just happened.** The same algorithm, the same code, two problems — and completely different behaviour. At $\kappa = 2$ the contours are nearly circular and the path runs almost straight to the bottom. At $\kappa = 25$ they form a canyon and the path **zigzags** across it, creeping along its length.

**The zigzag has a precise cause.** The gradient points perpendicular to the contour lines. In a circular bowl that is straight at the minimum. In an elongated canyon it points mostly *across* the valley rather than *along* it — so most of each step is spent bouncing between the walls, and only a small component makes real progress toward the bottom. Steepest descent is steepest *locally*, and locally steepest is not the direction you want.

**Note also that the step size is capped by the wrong eigenvalue.** With $\eta = 1/L$ set by the *largest* curvature, the step is chosen to be safe in the steep direction — and that same step is far too small for the shallow direction, where progress needs to happen. Both symptoms come from one number.

**Convert $\kappa$ into a cost, because that is what makes it real.** Error shrinks by $(1 - 1/\kappa)$ per step, so reducing it by $10^{-6}$ takes roughly:

| $\kappa$ | iterations |
|---|---|
| 2 | ~20 |
| 25 | ~338 |
| $10^4$ | ~138,000 |

Identical code, a 400× difference in cost between the last two rows, decided entirely by the **geometry of the problem** rather than by anything about the algorithm. That is why $\kappa$ is the villain of this workshop.

**And it makes the optimiser zoo comprehensible as one idea.** Every method you have heard of attacks $\kappa$:

- **Preconditioning** changes coordinates to round the bowl out.
- **Momentum** and **conjugate gradients** extract $\sqrt\kappa$ instead of $\kappa$ — at $\kappa = 10^4$ that is 100× fewer iterations ([Numerical Linear Algebra](../Numerical_Linear_Algebra/Numerical_Linear_Algebra.ipynb) S3).
- **Adam** approximates a per-coordinate rescaling, which is a cheap diagonal preconditioner.
- **Newton's method** uses the exact Hessian and achieves $\kappa = 1$, at cubic cost per step.

Four techniques, one motivation. Students who see them as answers to a single question retain far more than those who meet them as four unrelated recipes.

**And the same number appears throughout this curriculum.** It is [LMS](../../Intro_Time_Series/Intro_AdFilt_APA.ipynb)'s convergence penalty on correlated input, [RLS](../../Intro_Time_Series/Intro_RLS.ipynb)'s reason for carrying $R^{-1}$, and the digit-loss factor in [Numerical Linear Algebra](../Numerical_Linear_Algebra/Numerical_Linear_Algebra.ipynb). One eigenvalue ratio, four workshops.

In [ ]:
# Measured rate vs predicted (1 − μ/L) per step

# YOUR CODE HERE


This is *why* preconditioning, momentum, and Adam exist: they all attack $\kappa$. And it's the same $\lambda_{max}$ speed limit you met as $\mu < 2/\lambda_{max}$ in [LMS](../../Intro_Time_Series/Intro_AdFilt_APA.ipynb).

---
### 🕐 Session 3 of 4 — *Constraints: Lagrange & KKT* (~35 min)
**Goal:** optimize with equality and inequality constraints; read a Lagrangian like a force balance.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (SGD).

---

## 4. Constrained Optimization

💡 **Intuition.** At a constrained optimum you cannot improve *without leaving the feasible set* — so the objective's downhill direction must be exactly opposed by the constraint's 'wall'. Algebraically: $\nabla f$ is a combination of constraint normals. The multipliers $\lambda$ are the strengths of the wall forces — and economically, the *price* of tightening each constraint.

**Lagrange (equality).** For $\min f$ s.t. $g(x) = 0$: at a (regular) optimum $\exists \lambda$ with $\nabla f = \lambda \nabla g$. Solve by stationarity of $\mathcal{L}(x, \lambda) = f(x) - \lambda g(x)$.

**KKT (inequality, $h(x) \le 0$)** adds two conditions: $\lambda \ge 0$ (walls only push, never pull) and *complementary slackness* $\lambda h(x) = 0$ (an inactive constraint exerts no force).

**Worked example.** $\min\; x^T S x$ s.t. $\|x\| = 1$: stationarity gives $S x = \lambda x$ — the constrained minimizer is the **smallest eigenvector**. Eigenproblems *are* constrained optimization; this is how PCA, MVDR beamforming ([Array Processing](../../Intro_DSP/Array_Processing.ipynb)), and Rayleigh quotients arise.

In [ ]:
# Verify: minimize xᵀSx on the unit circle — numerically vs the eigen-answer

# YOUR CODE HERE


**What just happened.** A brute-force search over 2000 points on the unit circle found the minimiser $[-0.9864, 0.1643]$ with value $0.1221$. `eigh` returned the smallest eigenvector $[-0.9863, 0.1648]$ with eigenvalue $0.1221$. **Same point, same value — one found by searching, one by an eigen-decomposition.**

**The Lagrangian explains why they must agree.** Minimising $x^\top Sx$ subject to $\|x\|^2 = 1$ gives stationarity
$$2Sx = 2\lambda x \quad\Longrightarrow\quad Sx = \lambda x,$$
which is the eigenvalue equation. So the constrained minimiser *is* an eigenvector, and the multiplier $\lambda$ *is* the eigenvalue — which is also the optimal objective value, since $x^\top Sx = \lambda x^\top x = \lambda$ on the unit sphere.

**Let that land, because it reframes a familiar object.** Eigenproblems **are** constrained optimisation problems. Students have been computing eigenvectors since [Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) without a statement of what they optimise; the answer is "the quadratic form, on the unit sphere," with the smallest eigenvector minimising and the largest maximising.

That single fact explains a great deal of this curriculum at once. **PCA** maximises variance subject to unit norm. **MVDR beamforming** in [Array Processing](../../Intro_DSP/Array_Processing.ipynb) minimises output power subject to unit gain in the look direction, and its solution $R^{-1}a/(a^HR^{-1}a)$ drops straight out of the Lagrangian. The **Rayleigh quotient** in [Manifold Optimization](./Manifold_Optimization.ipynb) is the same problem solved by walking on the sphere instead of using a multiplier. Four workshops, one Lagrangian.

**On the small mismatch in the fourth decimal.** The two vectors differ by about $5\times10^{-4}$, and that is **grid resolution rather than error**: 2000 samples around the circle gives an angular spacing of $3.1\times10^{-3}$ radians, so the brute-force answer can be off by roughly half a grid step. Notice that the objective *values* agree exactly (0.1221 both) while the *locations* differ slightly — the characteristic pattern near any minimum, where the function is flat and so the value is far better determined than the argument. Worth remembering when you evaluate an optimiser: matching objective values is weaker evidence than matching arguments.

**And the multiplier has a reading beyond the algebra.** $\lambda$ measures how much the objective would improve per unit of constraint relaxation — the **shadow price** of the constraint. Physically, it is the strength of the wall force that stops the downhill walker from leaving the feasible set. That interpretation is what makes multipliers practically useful, and it is the same object that becomes the dual variable in [Convex Optimization II](./Convex_Optimization_2.ipynb).

---
### 🕐 Session 4 of 4 — *Stochastic Gradient Descent* (~40 min)
**Goal:** understand SGD's noise: why it's cheap, why it still converges, and why it can even help.
**Builds on:** Sessions 2–3.

---

## 5. SGD

💡 **Intuition.** Full gradients cost a pass over ALL data; SGD gambles on a mini-batch's estimate. The estimate is **unbiased** — right on average — so each step is downhill *in expectation*, and averaging over steps mimics averaging over data (the LLN from [Independence](../Analysis/Independence.ipynb)). The price: a noise floor set by step size × gradient variance. The classic cure: **decay the step size** — big steps to travel, small steps to settle. And in nonconvex landscapes the noise moonlights as an explorer, rattling the iterate out of narrow bad minima.

**The trade in one equation** (strongly convex case, stated): with constant step $\eta$,
$$E\|x_k - x^*\|^2 \lesssim \underbrace{(1 - \eta\mu)^k \|x_0 - x^*\|^2}_{\text{bias: shrinks}} + \underbrace{\frac{\eta \, \sigma^2}{\mu}}_{\text{noise floor: doesn't}}$$
Decaying $\eta_k \propto 1/k$ drives both terms to zero (at the slower $O(1/k)$ rate).

In [ ]:
# See the noise floor and the decay cure, on least squares

# YOUR CODE HERE


**What just happened.** Three step-size strategies, and the plot shows exactly the trade the theory predicts:

- **$\eta = 0.01$** — descends fast, then **flattens** at a visible floor and stops improving no matter how long it runs.
- **$\eta = 0.001$** — descends more slowly, and settles at a *lower* floor.
- **Decaying $\eta$** — matches the fast curve early and the low curve late.

**The floor is not a bug and it does not go away with patience.** The governing decomposition is
$$E\|x_k - x^*\|^2 \;\lesssim\; \underbrace{(1-\eta\mu)^k\|x_0-x^*\|^2}_{\text{bias — shrinks geometrically}} \;+\; \underbrace{\frac{\eta\sigma^2}{\mu}}_{\text{noise floor — constant}}.$$
The first term vanishes; the second does not depend on $k$ at all. So a constant step size buys you geometric progress *until* you reach a floor proportional to $\eta$, and then nothing. Halving $\eta$ halves the floor and halves the speed — which is precisely the gap between the first two curves.

**Which makes the decay schedule obvious rather than clever.** Large steps while far away (where speed matters and precision does not), small steps once close (where the floor matters and speed does not). $\eta_k \propto 1/(1+k/1000)$ drives both terms to zero, at the slower $O(1/k)$ rate — you give up geometric convergence in exchange for actually arriving.

**And this is the theory behind every learning-rate schedule you have seen.** Step decay, cosine annealing, warmup-then-decay: all of them are this equation. If you have ever watched a training loss plateau and then drop sharply the moment the learning rate was reduced, you were watching $\eta\sigma^2/\mu$ move down. It is not the model suddenly learning something new; it is the noise floor being lowered.

**Why SGD is worth the noise at all.** A full gradient costs a pass over *all* the data — on a million examples, a million evaluations for **one** step. A batch of 8 gives 125,000 steps for the same cost. The mini-batch gradient is **unbiased**, so each step is downhill in expectation and the errors average out over many steps ([the LLN](../Analysis/Independence.ipynb) again). SGD is not a crude approximation that happens to work; it is an unbiased estimator inside a method that tolerates unbiased noise.

**One honest note on the nonconvex claim.** The intuition cell says SGD's noise "moonlights as an explorer," rattling iterates out of narrow bad minima. That is well-supported empirically and *not* fully explained theoretically — the flat-minima-generalise-better link remains debated, and none of this session's proofs cover the nonconvex case. It is a plausible mechanism with good evidence, not a theorem, and [Training Dynamics](../../Intro_Mach_Learn/Training_Dynamics.ipynb) takes it up as an open question rather than a settled one.

## 6. Conclusion

Convexity is the promise, $\kappa$ the speed limit, Lagrange the wall forces, and SGD the affordable gamble with a noise floor you now know how to lower. Deep-learning practice ([Training Dynamics](../../Intro_Mach_Learn/Training_Dynamics.ipynb)) is engineering against exactly these quantities.

---
## Where next

- [Training Dynamics](../../Intro_Mach_Learn/Training_Dynamics.ipynb) — Adam, schedules, and regularization as applied versions of these ideas.
- [Adaptive Filtering](../../Intro_Time_Series/Intro_AdFilt_APA.ipynb) — SGD in real time, under the name LMS.
- [Estimation Theory](../Estimation_Theory/Estimation_Theory.ipynb) — what the minimum *means* statistically.